# 07: Matcher on K=50 candidates (compare with 05: K=20, test F0.5 0.925)
Pairs come from `06` (`cand_full_k50.parquet`). Split by Source 1 entity into train/val/test. Threshold is tuned on val for the competition metric: **macro F0.5 per Source 1 entity, singletons included**.

In [1]:
import os, sys, time
os.environ['TMP']=os.environ['TEMP']='D:/tmp'
sys.path.insert(0,'D:/Amazon_ML_Challenge')
import pandas as pd, numpy as np, lightgbm as lgb
from sklearn.metrics import roc_auc_score
from src.features import pair_features, add_group_features
W='D:/Amazon_ML_Challenge/work/'; TR='D:/Amazon_ML_Challenge/Hackathon/Datasets/Train Datasets/'
rd=lambda p,**k: pd.read_csv(p,sep='\t',dtype=str,keep_default_na=False,quoting=3,**k)
cand=pd.read_parquet(W+'cand_full_k50.parquet')
# one row per (s1, src, cand): cosines per channel + which channel retrieved it
P=cand.pivot_table(index=['s1_id','src','cand_id'],columns='channel',values='cos',aggfunc='max')
P=P.rename(columns={'name':'name_cos','addr':'addr_cos'}).reset_index()
P['in_name']=P.name_cos.notna().astype(int); P['in_addr']=P.addr_cos.notna().astype(int)
P[['name_cos','addr_cos']]=P[['name_cos','addr_cos']].fillna(0.0)
print(len(P),P.groupby('src').size().to_dict())

974751 {'s2': 487669, 's3': 487082}


In [2]:
# raw fields for the queries and candidate rows
gt=rd(TR+'train_ground_truth.tsv').sample(5000,random_state=0)   # same sample as notebook 04/06
need=set(P.s1_id)
s1=pd.concat([c[c.entity_id.isin(need)] for c in rd(TR+'train_source1.tsv',chunksize=500000)]).set_index('entity_id')
raw={}
for src in ('s2','s3'):
    ids=set(P.cand_id[P.src==src])
    raw[src]=pd.concat([c[c.entity_id.isin(ids)] for c in rd(TR+f'train_source{src[1]}.tsv',chunksize=500000)]).set_index('entity_id')
parts=[]
for src in ('s2','s3'):
    p=P[P.src==src].copy(); b=raw[src].loc[p.cand_id]
    p['b_name']=b.business_name.values; p['b_addr']=b.business_address.values; p['b_country']=b.country.values
    parts.append(p)
P=pd.concat(parts,ignore_index=True)
a=s1.loc[P.s1_id]; P['a_name']=a.business_name.values; P['a_addr']=a.business_address.values; P['a_country']=a.country.values
truth=gt.set_index('source1_entity_id').matched_entity_ids.map(lambda s:set(x for x in s.split(',') if x))
P['label']=[int(c in truth[s]) for s,c in zip(P.s1_id,P.cand_id)]
print(P.label.mean(),P.label.sum())

0.016776592175847985 16353


In [3]:
t=time.time(); F=pair_features(P); F=add_group_features(F,P[['s1_id','src']]); print(F.shape,round(time.time()-t),'s')
F['src_is_s3']=(P.src=='s3').astype(int)

(974751, 49) 158 s


## Train / val / test split by Source 1 entity

In [4]:
ents=np.array(sorted(need)); rng=np.random.RandomState(0); rng.shuffle(ents)
n=len(ents); split={**{e:'train' for e in ents[:int(.6*n)]},**{e:'val' for e in ents[int(.6*n):int(.8*n)]},**{e:'test' for e in ents[int(.8*n):]}}
P['split']=P.s1_id.map(split)
tr,va,te=[P.split==s for s in ('train','val','test')]
m=lgb.LGBMClassifier(n_estimators=2000,learning_rate=0.03,num_leaves=63,subsample=0.8,subsample_freq=1,colsample_bytree=0.8,min_child_samples=20,verbose=-1)
m.fit(F[tr],P.label[tr],eval_set=[(F[va],P.label[va])],callbacks=[lgb.early_stopping(100,verbose=False)])
P['p']=m.predict_proba(F)[:,1]
print('best iter',m.best_iteration_,'val AUC',round(roc_auc_score(P.label[va],P.p[va]),4),'test AUC',round(roc_auc_score(P.label[te],P.p[te]),4))
pd.Series(m.feature_importances_,F.columns).sort_values(ascending=False).head(15)

D:\Amazon_ML_Challenge\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


best iter 420 val AUC 0.9996 test AUC 0.9996


n_raw_ratio     1915
n_lenratio      1174
addr_cos_rel    1148
n_ntok_b         907
num_jac          887
a_tset           879
a_ntok_a         875
a_jac_rel        871
a_jac_rank       860
a_jac            838
a_ntok_b         791
a_ratio          781
a_tsort          707
a_part           670
a_tset_rel       659
dtype: int32

## Metric: macro F0.5 per Source 1 entity (singletons included)
Entities with no candidate above the threshold predict an empty list. Also shows the ceiling if every candidate were classified perfectly (blocking recall limit).

In [5]:
def f05(pred,tru):
    if not pred and not tru: return 1.0
    if not pred or not tru: return 0.0
    h=len(pred&tru); p,r=h/len(pred),h/len(tru)
    return 0.0 if h==0 else 1.25*p*r/(0.25*p+r)
def macro(split_name,thr,col='p'):
    d=P[P.split==split_name]; sel=d[d[col]>=thr].groupby('s1_id').cand_id.apply(set)
    ents_=[e for e in ents if split[e]==split_name]
    return np.mean([f05(sel.get(e,set()),truth[e]) for e in ents_])
def oracle(split_name):
    d=P[(P.split==split_name)&(P.label==1)].groupby('s1_id').cand_id.apply(set)
    ents_=[e for e in ents if split[e]==split_name]
    return np.mean([f05(d.get(e,set()),truth[e]) for e in ents_])
grid=np.arange(0.2,0.96,0.05); res=pd.DataFrame({'thr':grid,'val':[macro('val',t) for t in grid]})
best=res.loc[res.val.idxmax()]; print(res.round(4).to_string(index=False))
print('best thr',best.thr,'val F0.5',round(best.val,4),'| test F0.5 @thr',round(macro('test',best.thr),4),'| oracle val',round(oracle('val'),4),'oracle test',round(oracle('test'),4))
print('share of singletons in test:',np.mean([len(truth[e])==0 for e in ents if split[e]=='test']).round(3))

 thr    val
0.20 0.8811
0.25 0.8924
0.30 0.9015
0.35 0.9092
0.40 0.9173
0.45 0.9195
0.50 0.9214
0.55 0.9233
0.60 0.9255
0.65 0.9262
0.70 0.9253
0.75 0.9253
0.80 0.9244
0.85 0.9203
0.90 0.9163
0.95 0.9020


best thr 0.6499999999999999 val F0.5 0.9262 | test F0.5 @thr 0.9258 | oracle val 0.9872 oracle test 0.986
share of singletons in test: 0.059


In [6]:
m.booster_.save_model(W+'matcher_lgbm_k50.txt'); pd.Series({'threshold':float(best.thr)}).to_json(W+'matcher_meta_k50.json')
P[['s1_id','src','cand_id','p','label','split']].to_parquet(W+'scored_pairs_k50.parquet'); print('saved')

saved
